# 01 - Laws Preprocessing

Questo notebook mostra il passaggio dal corpus HTML originale a un dataset pulito per retrieval e graph-aware RAG. La trasformazione resta nel codice riusabile in `src/legal_rag/law_preprocessing`; qui eseguiamo la pipeline e ispezioniamo esempi concreti prima/dopo.

In [1]:
from pathlib import Path
import html
import json
import sys

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from legal_rag.laws_preprocessing import LawsPreprocessingConfig, run_laws_preprocessing

SOURCE_DIR = ROOT / "data" / "laws_html"
OUTPUT_DIR = ROOT / "data" / "laws_dataset_clean"

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

## 1. Run deterministico

La pipeline legge solo `data/laws_html/`, ignora i file non sorgente, scrive in una directory temporanea e sostituisce `data/laws_dataset_clean/` solo dopo avere superato le validazioni. Il corpus HTML non viene modificato.

In [2]:
manifest = run_laws_preprocessing(
    LawsPreprocessingConfig(
        source_dir=str(SOURCE_DIR),
        output_dir=str(OUTPUT_DIR),
        chunk_size=600,
        chunk_overlap=80,
        strict=False,
    )
)

pd.DataFrame([
    {
        "ready_for_indexing": manifest["ready_for_indexing"],
        "source_hash": manifest["source_hash"],
        "valid_html_files": manifest["inventory"]["valid_html_files"],
        "ignored_files": manifest["inventory"]["ignored_files"],
        "unresolved_refs": manifest["unresolved_refs"],
    }
])

,ready_for_indexing,source_hash,valid_html_files,ignored_files,unresolved_refs
0,True,aa46ea3758c1de5596a8902b95e90cdfc6d08fbc2956086495a020680be22459,3145,[.DS_Store],277


## 2. Caricamento degli artifact

Gli output principali sono JSONL ordinati per ID stabile. Usiamo `pandas` solo per ispezione: la pipeline non dipende da notebook o DataFrame.

In [3]:
def read_jsonl_df(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_json(path, lines=True)

laws_df = read_jsonl_df(OUTPUT_DIR / "laws.jsonl")
articles_df = read_jsonl_df(OUTPUT_DIR / "articles.jsonl")
passages_df = read_jsonl_df(OUTPUT_DIR / "passages.jsonl")
notes_df = read_jsonl_df(OUTPUT_DIR / "notes.jsonl")
edges_df = read_jsonl_df(OUTPUT_DIR / "edges.jsonl")
chunks_df = read_jsonl_df(OUTPUT_DIR / "chunks.jsonl")
profile = json.loads((OUTPUT_DIR / "dataset_profile.json").read_text(encoding="utf-8"))

counts_df = pd.DataFrame([manifest["counts"]]).T.rename(columns={0: "rows"})
counts_df

,rows
laws,3145
articles,17774
passages,76390
notes,8380
edges,35159
chunks,76467


## 3. Che cosa viene ricavato

Ogni componente nasce da un segnale deterministico nel file HTML o nel nome file. La tabella sotto riassume cosa rappresenta e come viene derivato.

In [4]:
componenti = pd.DataFrame(
    [
        {
            "componente": "law",
            "rappresenta": "Una legge regionale del corpus.",
            "come viene ricavata": "Dal filename per data/numero/ID stabile e dagli heading HTML per il titolo; il preambolo viene preso dai blocchi prima del primo articolo.",
            "output": "laws.jsonl",
        },
        {
            "componente": "article",
            "rappresenta": "Un articolo giuridico dentro una legge.",
            "come viene ricavata": "Da anchor HTML come `name=articolo_...` o da righe testuali `Art. N`; l'indice iniziale viene saltato.",
            "output": "articles.jsonl",
        },
        {
            "componente": "passage",
            "rappresenta": "Una porzione strutturata di articolo: intro, comma, lettera.",
            "come viene ricavata": "Dalle righe dell'articolo: `1.` apre un comma, `a)` apre una lettera, il testo prima del primo comma diventa intro.",
            "output": "passages.jsonl",
        },
        {
            "componente": "note",
            "rappresenta": "Una nota redazionale collegata a testo, articolo o passaggio.",
            "come viene ricavata": "Da anchor come `name=nota_...` e link interni `href=#nota_...`; conserva testo e link espliciti.",
            "output": "notes.jsonl",
        },
        {
            "componente": "edge",
            "rappresenta": "Una relazione esplicita tra leggi o articoli.",
            "come viene ricavata": "Da hyperlink con `numero_legge` e citazioni testuali risolte nel corpus; il tipo relazione viene classificato da evidenza testuale esplicita.",
            "output": "edges.jsonl",
        },
        {
            "componente": "chunk",
            "rappresenta": "Unita pronta per embedding e retrieval.",
            "come viene ricavata": "Da ogni passage con chunking word-based deterministico; include metadati denormalizzati per filtri, citazioni e grafo.",
            "output": "chunks.jsonl",
        },
    ]
)
componenti

,componente,rappresenta,come viene ricavata,output
0,law,Una legge regionale del corpus.,Dal filename per data/numero/ID stabile e dagli heading HTML per il titolo; il preambolo viene preso dai blocchi prima del primo articolo.,laws.jsonl
1,article,Un articolo giuridico dentro una legge.,Da anchor HTML come `name=articolo_...` o da righe testuali `Art. N`; l'indice iniziale viene saltato.,articles.jsonl
2,passage,"Una porzione strutturata di articolo: intro, comma, lettera.","Dalle righe dell'articolo: `1.` apre un comma, `a)` apre una lettera, il testo prima del primo comma diventa intro.",passages.jsonl
3,note,"Una nota redazionale collegata a testo, articolo o passaggio.",Da anchor come `name=nota_...` e link interni `href=#nota_...`; conserva testo e link espliciti.,notes.jsonl
4,edge,Una relazione esplicita tra leggi o articoli.,Da hyperlink con `numero_legge` e citazioni testuali risolte nel corpus; il tipo relazione viene classificato da evidenza testuale espli...,edges.jsonl
5,chunk,Unita pronta per embedding e retrieval.,"Da ogni passage con chunking word-based deterministico; include metadati denormalizzati per filtri, citazioni e grafo.",chunks.jsonl


## 4. Quality gates

Questi controlli impediscono di produrre un dataset sporco: ID duplicati, chunk incompleti, liste serializzate come stringhe, self-loop nel grafo clean o artifact mancanti fanno fallire il run.

In [5]:
quality_df = pd.DataFrame(
    [{"gate": gate, "passed": ok} for gate, ok in manifest["quality_gates"].items()]
).sort_values("gate")

diagnostics_df = pd.DataFrame(
    [
        {"metric": "ignored_files", "value": manifest["inventory"]["ignored_files"]},
        {"metric": "unresolved_refs", "value": manifest["unresolved_refs"]},
        {"metric": "dropped_duplicates", "value": manifest["dropped_duplicates"]},
        {"metric": "ready_for_indexing", "value": manifest["ready_for_indexing"]},
    ]
)

display(quality_df)
display(diagnostics_df)

,gate,passed
8,chunks_jsonl_non_empty,True
4,clean_graph_edges_have_no_self_loops,True
6,law_statuses_allowed,True
3,list_metadata_fields_are_lists,True
7,manifest_output_files_exist_and_hash,True
5,relation_types_allowed,True
2,required_chunk_fields_present,True
1,stable_ids_non_empty_duplicate_free,True
0,valid_source_html_found,True


,metric,value
0,ignored_files,[.DS_Store]
1,unresolved_refs,277
2,dropped_duplicates,"{'laws': 0, 'articles': 0, 'passages': 0, 'notes': 0, 'edges': 0, 'chunks': 0}"
3,ready_for_indexing,True


## 5. Prima e dopo su una legge reale

Scegliamo in modo deterministico una legge con almeno una relazione esplicita, cosi si vede l'intero passaggio: HTML sorgente, legge normalizzata, articoli, passaggi, chunk e grafo.

In [6]:
preferred_source_file = "10001_LR-22-dicembre-2021-n35.html"

if preferred_source_file in set(laws_df["source_file"]):
    example_law = laws_df.loc[laws_df["source_file"] == preferred_source_file].iloc[0]
else:
    src_law_id = edges_df.sort_values(["relation_type", "src_law_id"]).iloc[0]["src_law_id"]
    example_law = laws_df.loc[laws_df["law_id"] == src_law_id].iloc[0]

example_law_id = example_law["law_id"]
example_source_file = example_law["source_file"]
example_html_path = SOURCE_DIR / example_source_file
example_html = example_html_path.read_text(encoding="utf-8", errors="replace")

pd.DataFrame(
    [
        {
            "law_id": example_law_id,
            "source_file": example_source_file,
            "law_title": example_law["law_title"],
            "law_status": example_law["law_status"],
            "html_chars": len(example_html),
        }
    ]
)

,law_id,source_file,law_title,law_status,html_chars
0,vda:lr:2021-12-22:35,10001_LR-22-dicembre-2021-n35.html,"Legge regionale 22 dicembre 2021, n. 35 - Testo vigente",current,139393


### Prima: frammento HTML originale

Il file sorgente contiene heading, paragrafi, anchor interne, link ad altre leggi e spesso un indice. La pipeline conserva i segnali utili e rimuove la complessita HTML dal contratto dati.

In [7]:
snippet = example_html[:5000]
display(HTML(f"<p><strong>File:</strong> {html.escape(str(example_html_path))}</p>"))
display(HTML("<pre style='white-space: pre-wrap; max-height: 520px; overflow: auto; border: 1px solid #ddd; padding: 12px;'>" + html.escape(snippet) + "</pre>"))

### Dopo: record `law`

Il record legge stabilizza identita, titolo, status, preambolo e provenienza. `law_id` deriva da data e numero, non dal path assoluto.

In [8]:
law_columns = ["law_id", "law_date", "law_number", "law_title", "law_status", "status_confidence", "source_file"]
laws_df.loc[laws_df["law_id"] == example_law_id, law_columns]

,law_id,law_date,law_number,law_title,law_status,status_confidence,source_file
3036,vda:lr:2021-12-22:35,2021-12-22,35,"Legge regionale 22 dicembre 2021, n. 35 - Testo vigente",current,0.72,10001_LR-22-dicembre-2021-n35.html


### Dopo: articoli e passaggi

Gli articoli vengono trovati dagli anchor `articolo_*` o da righe `Art. N`. I passaggi sono segmenti piu piccoli: intro, commi e lettere. Questo evita che il chunking tagli alla cieca la struttura legale.

In [9]:
def clip(value, n=180):
    text = "" if value is None else str(value)
    return text if len(text) <= n else text[:n] + "..."

example_articles = articles_df.loc[articles_df["law_id"] == example_law_id].copy()
example_passages = passages_df.loc[passages_df["law_id"] == example_law_id].copy()

article_view = example_articles[["article_id", "article_label_norm", "article_heading", "article_status", "structure_path"]].head(8)
passage_view = example_passages[["passage_id", "article_id", "passage_label", "passage_kind", "passage_text", "related_law_ids", "relation_types"]].head(8).copy()
passage_view["passage_text"] = passage_view["passage_text"].map(lambda value: clip(value, 220))

display(article_view)
display(passage_view)

,article_id,article_label_norm,article_heading,article_status,structure_path
16240,vda:lr:2021-12-22:35#art:1,1,NaN,current,CAPO I
16241,vda:lr:2021-12-22:35#art:10,10,NaN,current,CAPO II
16242,vda:lr:2021-12-22:35#art:11,11,NaN,current,CAPO II
16243,vda:lr:2021-12-22:35#art:12,12,NaN,past,CAPO II
16244,vda:lr:2021-12-22:35#art:13,13,NaN,current,CAPO III
16245,vda:lr:2021-12-22:35#art:14,14,NaN,current,CAPO III
16246,vda:lr:2021-12-22:35#art:15,15,NaN,current,CAPO III
16247,vda:lr:2021-12-22:35#art:16,16,NaN,current,CAPO III


,passage_id,article_id,passage_label,passage_kind,passage_text,related_law_ids,relation_types
68309,vda:lr:2021-12-22:35#art:1#p:c1,vda:lr:2021-12-22:35#art:1,c1,comma,"1. Per il periodo di imposta 2022, i soggetti con reddito complessivo, determinato ai fini dell'imposta sul reddito delle persone fisich...",[],[]
68310,vda:lr:2021-12-22:35#art:1#p:c2,vda:lr:2021-12-22:35#art:1,c2,comma,"2. L'onere, in termini di minore entrata, derivante dall'applicazione del presente articolo è determinato in euro 2.500.000 a valere sul...",[],[]
68311,vda:lr:2021-12-22:35#art:1#p:intro,vda:lr:2021-12-22:35#art:1,intro,intro,(Esenzione dall'addizionale regionale all'imposta sul reddito delle persone fisiche (IRPEF) per l'anno 2022),[],[]
68312,vda:lr:2021-12-22:35#art:10#p:c1,vda:lr:2021-12-22:35#art:10,c1,comma,"1. Per l'anno 2022, entro il 15 marzo 2022 gli enti di cui all'articolo 1 della l.r. 22/2010 comunicano alla struttura regionale compete...","[vda:lr:2010-07-23:22, vda:lr:2014-08-05:6]",[REFERENCES]
68313,vda:lr:2021-12-22:35#art:10#p:c2,vda:lr:2021-12-22:35#art:10,c2,comma,"2. Per l'anno 2022 continuano a trovare applicazione le disposizioni di cui ai commi 8bis, 8ter, 8quater, 8quinquies e 9 dell'articolo 3...",[vda:lr:2020-12-21:12],[REFERENCES]
68314,vda:lr:2021-12-22:35#art:10#p:c3,vda:lr:2021-12-22:35#art:10,c3,comma,"3. Per l'anno 2022, per lo svolgimento delle procedure concorsuali, selettive uniche, selettive interne e degli accertamenti linguistici...",[vda:lr:2010-07-23:22],[REFERENCES]
68315,vda:lr:2021-12-22:35#art:10#p:c4,vda:lr:2021-12-22:35#art:10,c4,comma,"4. Per l'anno 2022, le commissioni esaminatrici di cui all'articolo 36 del r.r. 1/2013 possono svolgere i propri lavori in modalità tele...",[],[]
68316,vda:lr:2021-12-22:35#art:10#p:c5,vda:lr:2021-12-22:35#art:10,c5,comma,"5. Le disposizioni in materia di procedure selettive interne di cui all'articolo 5bis della legge regionale 22 dicembre 2017, n. 21 (Leg...",[vda:lr:2017-12-22:21],[REFERENCES]


### Dopo: chunk RAG-ready

Ogni chunk mantiene il testo da recuperare, il testo arricchito per embedding e i metadati necessari per filtri, citazioni e retrieval graph-aware.

In [10]:
example_chunks = chunks_df.loc[chunks_df["law_id"] == example_law_id].copy()
chunk_view = example_chunks[
    [
        "chunk_id",
        "article_label_norm",
        "passage_label",
        "law_status",
        "article_status",
        "index_views",
        "related_law_ids",
        "inbound_law_ids",
        "outbound_law_ids",
        "relation_types",
        "text",
    ]
].head(5).copy()
chunk_view["text"] = chunk_view["text"].map(lambda value: clip(value, 260))
chunk_view

,chunk_id,article_label_norm,passage_label,law_status,article_status,index_views,related_law_ids,inbound_law_ids,outbound_law_ids,relation_types,text
68386,vda:lr:2021-12-22:35#art:1#p:c1#chunk:0,1,c1,current,current,"[historical, current]",[],"[vda:lr:1994-08-27:64, vda:lr:2000-01-25:5, vda:lr:2002-05-20:5, vda:lr:2006-05-19:11, vda:lr:2008-04-15:9, vda:lr:2009-08-04:30, vda:lr...","[vda:lr:1980-07-22:34, vda:lr:1982-06-28:16, vda:lr:1984-07-06:33, vda:lr:1990-11-27:75, vda:lr:1991-07-30:26, vda:lr:1991-08-31:37, vda...","[ABROGATED_BY, ABROGATES, AMENDS, INSERTS, MODIFIED_BY, REFERENCES, REPLACES]","1. Per il periodo di imposta 2022, i soggetti con reddito complessivo, determinato ai fini dell'imposta sul reddito delle persone fisich..."
68387,vda:lr:2021-12-22:35#art:1#p:c2#chunk:0,1,c2,current,current,"[historical, current]",[],"[vda:lr:1994-08-27:64, vda:lr:2000-01-25:5, vda:lr:2002-05-20:5, vda:lr:2006-05-19:11, vda:lr:2008-04-15:9, vda:lr:2009-08-04:30, vda:lr...","[vda:lr:1980-07-22:34, vda:lr:1982-06-28:16, vda:lr:1984-07-06:33, vda:lr:1990-11-27:75, vda:lr:1991-07-30:26, vda:lr:1991-08-31:37, vda...","[ABROGATED_BY, ABROGATES, AMENDS, INSERTS, MODIFIED_BY, REFERENCES, REPLACES]","2. L'onere, in termini di minore entrata, derivante dall'applicazione del presente articolo è determinato in euro 2.500.000 a valere sul..."
68388,vda:lr:2021-12-22:35#art:1#p:intro#chunk:0,1,intro,current,current,"[historical, current]",[],"[vda:lr:1994-08-27:64, vda:lr:2000-01-25:5, vda:lr:2002-05-20:5, vda:lr:2006-05-19:11, vda:lr:2008-04-15:9, vda:lr:2009-08-04:30, vda:lr...","[vda:lr:1980-07-22:34, vda:lr:1982-06-28:16, vda:lr:1984-07-06:33, vda:lr:1990-11-27:75, vda:lr:1991-07-30:26, vda:lr:1991-08-31:37, vda...","[ABROGATED_BY, ABROGATES, AMENDS, INSERTS, MODIFIED_BY, REFERENCES, REPLACES]",(Esenzione dall'addizionale regionale all'imposta sul reddito delle persone fisiche (IRPEF) per l'anno 2022)
68389,vda:lr:2021-12-22:35#art:10#p:c1#chunk:0,10,c1,current,current,"[historical, current]","[vda:lr:2010-07-23:22, vda:lr:2014-08-05:6]","[vda:lr:1994-08-27:64, vda:lr:2000-01-25:5, vda:lr:2002-05-20:5, vda:lr:2006-05-19:11, vda:lr:2008-04-15:9, vda:lr:2009-08-04:30, vda:lr...","[vda:lr:1980-07-22:34, vda:lr:1982-06-28:16, vda:lr:1984-07-06:33, vda:lr:1990-11-27:75, vda:lr:1991-07-30:26, vda:lr:1991-08-31:37, vda...","[ABROGATED_BY, ABROGATES, AMENDS, INSERTS, MODIFIED_BY, REFERENCES, REPLACES]","1. Per l'anno 2022, entro il 15 marzo 2022 gli enti di cui all'articolo 1 della l.r. 22/2010 comunicano alla struttura regionale compete..."
68390,vda:lr:2021-12-22:35#art:10#p:c2#chunk:0,10,c2,current,current,"[historical, current]",[vda:lr:2020-12-21:12],"[vda:lr:1994-08-27:64, vda:lr:2000-01-25:5, vda:lr:2002-05-20:5, vda:lr:2006-05-19:11, vda:lr:2008-04-15:9, vda:lr:2009-08-04:30, vda:lr...","[vda:lr:1980-07-22:34, vda:lr:1982-06-28:16, vda:lr:1984-07-06:33, vda:lr:1990-11-27:75, vda:lr:1991-07-30:26, vda:lr:1991-08-31:37, vda...","[ABROGATED_BY, ABROGATES, AMENDS, INSERTS, MODIFIED_BY, REFERENCES, REPLACES]","2. Per l'anno 2022 continuano a trovare applicazione le disposizioni di cui ai commi 8bis, 8ter, 8quater, 8quinquies e 9 dell'articolo 3..."


### `text` vs `text_for_embedding`

`text` resta il contenuto pulito del passaggio. `text_for_embedding` aggiunge contesto legale prima del testo, cosi un embedding sa a quale legge, articolo e struttura appartiene il contenuto.

In [11]:
sample_chunk = example_chunks.iloc[0]
display(Markdown("**text**"))
display(HTML("<pre style='white-space: pre-wrap; border: 1px solid #ddd; padding: 12px;'>" + html.escape(sample_chunk["text"][:900]) + "</pre>"))
display(Markdown("**text_for_embedding**"))
display(HTML("<pre style='white-space: pre-wrap; border: 1px solid #ddd; padding: 12px;'>" + html.escape(sample_chunk["text_for_embedding"][:1200]) + "</pre>"))

**text**

**text_for_embedding**

### Dopo: relazioni esplicite

Le relazioni non sono inferite semanticamente. Nascono solo da link o citazioni testuali risolte nel corpus, e conservano una evidenza ispezionabile.

In [12]:
example_edges = edges_df.loc[edges_df["src_law_id"] == example_law_id].copy()
edge_columns = ["relation_type", "src_article_id", "src_passage_id", "dst_law_id", "dst_article_label_norm", "extraction_method", "confidence", "evidence"]
edge_view = example_edges[edge_columns].head(10).copy()
if not edge_view.empty:
    edge_view["evidence"] = edge_view["evidence"].map(lambda value: clip(value, 260))
edge_view

,relation_type,src_article_id,src_passage_id,dst_law_id,dst_article_label_norm,extraction_method,confidence,evidence
51,REFERENCES,vda:lr:2021-12-22:35#art:14,vda:lr:2021-12-22:35#art:14#p:c1,vda:lr:1998-12-07:54,NaN,href,0.45,"1. A decorrere dall'anno 2022, in deroga all'articolo 99, comma 2, ultimo periodo, della legge regionale 7 dicembre 1998, n. 54 (Sistema..."
317,AMENDS,vda:lr:2021-12-22:35#art:5,vda:lr:2021-12-22:35#art:5#p:c5,vda:lr:2010-07-23:22,NaN,href,0.70,"5. L'Amministrazione regionale e gli altri enti di cui all'articolo 1 della legge regionale 23 luglio 2010, n. 22 (Nuova disciplina dell..."
588,AMENDS,vda:lr:2021-12-22:35#art:3,vda:lr:2021-12-22:35#art:3#p:intro,vda:lr:2008-04-15:9,NaN,href,0.70,"(Disposizioni in materia di tasse automobilistiche. Modificazioni alla legge regionale 15 aprile 2008, n. 9)"
925,REFERENCES,vda:lr:2021-12-22:35#art:30,vda:lr:2021-12-22:35#art:30#p:c3,vda:lr:2020-12-21:12,NaN,href,0.45,"""5. Al finanziamento dell'onere di cui al presente articolo si provvede, per l'anno 2021, in deroga a quanto previsto dalla legge region..."
1085,REFERENCES,vda:lr:2021-12-22:35#art:30,vda:lr:2021-12-22:35#art:30#p:c4,vda:lr:2021-08-05:24,NaN,href,0.45,4. Il maggior onere derivante dall'applicazione del presente articolo è determinato in annui euro 50.000 a decorrere dall'anno 2022 (Mis...
1311,REFERENCES,vda:lr:2021-12-22:35#art:9,vda:lr:2021-12-22:35#art:9#p:c9,vda:lr:2020-02-11:1,NaN,href,0.45,"9. Per gli incarichi di particolare posizione organizzativa in essere alla data di entrata in vigore della presente legge, il termine de..."
1549,REFERENCES,vda:lr:2021-12-22:35#art:26,vda:lr:2021-12-22:35#art:26#p:c1.lit_a,vda:lr:1996-07-10:13,NaN,href,0.45,"a) imprese che svolgono attività di somministrazione di alimenti e bevande di cui alla legge regionale 3 gennaio 2006, n. 1 (Disciplina ..."
1734,REFERENCES,vda:lr:2021-12-22:35#art:36,vda:lr:2021-12-22:35#art:36#p:intro,vda:lr:2010-12-10:40,NaN,href,0.45,(Iscrizione contabile delle quote di ammortamento dell'indebitamento contratto ai sensi dell'articolo 40 della legge regionale 10 dicemb...
2058,REFERENCES,vda:lr:2021-12-22:35#art:9,vda:lr:2021-12-22:35#art:9#p:c1,vda:lr:2010-07-23:22,NaN,href,0.45,"1. Al comma 5 dell'articolo 20 della l.r. 22/2010, dopo le parole: ""di particolare e comprovata qualificazione professionale"", sono inse..."
2103,REFERENCES,vda:lr:2021-12-22:35#art:17,vda:lr:2021-12-22:35#art:17#p:c5,vda:lr:2017-07-31:11,NaN,href,0.45,"5. Il finanziamento di cui al comma 2, lettera c), è destinato in via esclusiva e vincolata al finanziamento da parte dell'Azienda USL d..."


### Dopo: note

Le note sono importanti per status e modifiche: spesso indicano abrogazioni, sostituzioni o inserimenti. Il dataset le conserva separatamente e le collega agli articoli/passaggi quando il testo contiene link interni alla nota.

In [13]:
example_notes = notes_df.loc[notes_df["law_id"] == example_law_id].copy()
note_columns = ["note_id", "note_anchor_name", "note_kind", "linked_article_ids", "linked_passage_ids", "note_text"]
note_view = example_notes[note_columns].head(8).copy() if not example_notes.empty else pd.DataFrame(columns=note_columns)
if not note_view.empty:
    note_view["note_text"] = note_view["note_text"].map(lambda value: clip(value, 260))
note_view

,note_id,note_anchor_name,note_kind,linked_article_ids,linked_passage_ids,note_text
8257,vda:lr:2021-12-22:35#note:nota_1,nota_1,modified,[vda:lr:2021-12-22:35#art:9],[vda:lr:2021-12-22:35#art:9#p:c8],"Comma modificato dal comma 1 dell'articolo 71 della L.R. 1° agosto 2022, n. 18.\nNella formulazione originaria, il comma 8 dell'articolo..."
8258,vda:lr:2021-12-22:35#note:nota_2,nota_2,modified,[vda:lr:2021-12-22:35#art:10],[vda:lr:2021-12-22:35#art:10#p:c3],"Comma modificato dalla lettera a) del comma 1 dell'articolo 3 della L.R. 27 maggio 2022, n. 6.\nNella formulazione originaria, il comma ..."
8259,vda:lr:2021-12-22:35#note:nota_3,nota_3,modified,[vda:lr:2021-12-22:35#art:10],[vda:lr:2021-12-22:35#art:10#p:c4],"Comma modificato dalla lettera b) del comma 1 dell'articolo 3 della L.R. 27 maggio 2022, n. 6.\nNella formulazione originaria, il comma ..."
8260,vda:lr:2021-12-22:35#note:nota_3a,nota_3a,abrogated,[vda:lr:2021-12-22:35#art:12],[vda:lr:2021-12-22:35#art:12#p:intro],"Articolo abrogato dal comma 7 dell'articolo 8 della legge regionale 19 dicembre 2023, n. 25.\nNella formulazione originaria, l'articolo ..."
8261,vda:lr:2021-12-22:35#note:nota_4,nota_4,abrogated,[],[],"Articolo abrogato dal comma 2 dell'articolo 2 della legge regionale 19 dicembre 2023, n. 25.\nIl comma 1 dell'articolo 14 era già stato ..."
8262,vda:lr:2021-12-22:35#note:nota_5,nota_5,modified,[vda:lr:2021-12-22:35#art:17],[vda:lr:2021-12-22:35#art:17#p:c3.lit_d],"Lettera modificata dal comma 1 dell'articolo 3 della L.R. 25 ottobre 2022, n. 22.\nNella formulazione originaria, la lettera d) del comm..."
8263,vda:lr:2021-12-22:35#note:nota_6,nota_6,abrogated,[],[],"Articolo abrogato dal comma 2 dell'articolo 3 della L.R. 25 ottobre 2022, n. 22.\nNella formulazione originaria, l'articolo 18 recitava:"
8264,vda:lr:2021-12-22:35#note:nota_7,nota_7,abrogated,[],[],"Articolo abrogato dalla lettera g) del comma 1 dell'articolo 15 della L.R. 2 aprile 2025, n. 8.\nNella formulazione originaria, l'art. 2..."


## 6. Profilo dataset

`dataset_profile.json` raccoglie un riepilogo compatto per i notebook successivi: conteggi, distribuzioni e piccoli esempi.

In [14]:
status_distribution = pd.Series(profile["status_distribution"], name="laws").sort_index().to_frame()
relation_distribution = pd.Series(profile["relation_type_distribution"], name="edges").sort_values(ascending=False).to_frame()

display(status_distribution)
display(relation_distribution)

,laws
current,2404
past,730
unknown,11


,edges
REFERENCES,13295
AMENDS,6283
ABROGATED_BY,5984
REPLACES,2715
MODIFIED_BY,2378
REPLACED_BY,2223
INSERTS,1371
INSERTED_BY,720
ABROGATES,190
